# DFTR-Filter: constrained example

This notebook demonstrates `py_trsqp.DFTRFilter` on an **output-constrained**
problem, where the constraint is a black box handled by the method's filter
and (when needed) restoration phase.

We minimise the same quadratic subject to a unit-disc inequality:

$$\min_{x}\; 10\,(x_1 - 1)^2 + 4\,(x_2 - 1)^2
\quad\text{s.t.}\quad 1 - x_1^2 - x_2^2 \ge 0.$$

The unconstrained minimiser $(1, 1)$ is **infeasible** (it lies outside the
unit disc), so the constrained optimum sits on the constraint boundary --
a good visual test of constraint handling. Inequalities follow the paper's
convention $c_i(x) \ge 0$.

In [ ]:
# --- make py_trsqp importable and load the local plotting helpers ---
# This notebook assumes it is run from the `examples/` directory (the Jupyter
# default). If py_trsqp is pip-installed you can drop the sys.path lines.
import os, sys

sys.path.insert(0, os.path.abspath(".."))   # repo root -> `import py_trsqp`
sys.path.insert(0, os.path.abspath("."))    # examples/ -> `import viz`

import numpy as np
import matplotlib.pyplot as plt

from py_trsqp import DFTRFilter
import viz

### Define the objective and constraint

In [ ]:
def cf(x):
    return 10 * (x[0] - 1.0) ** 2 + 4 * (x[1] - 1.0) ** 2

def disc(x):
    """Inequality constraint c(x) >= 0: the closed unit disc."""
    return 1.0 - (x[0] ** 2 + x[1] ** 2)

### Run the optimizer

Inequality constraints are passed as a list of callables via `ineqcs`
(equality constraints would go in `eqcs`). Each is a black box: no
derivatives are supplied.

In [ ]:
opt = DFTRFilter(
    x0=[-2.0, -2.0],
    cf=cf,
    lb=-3.0,
    ub=3.0,
    ineqcs=[disc],
)
opt.optimize(max_iter=200)

print(viz.summary(opt))

The reported `best viol.` should be ~0 (feasible), and the best point should
lie on the unit circle. Compare against SciPy's SLSQP as a sanity check:

In [ ]:
from scipy.optimize import minimize

ref = minimize(cf, [0.8, 0.6],
               constraints=[{"type": "ineq", "fun": disc}])
print("DFTR-Filter :", opt.best.x, " f =", opt.best.f)
print("SciPy SLSQP :", ref.x, " f =", ref.fun)
print("||x0^2+x1^2 - 1|| at DFTR best:",
      abs(opt.best.x[0] ** 2 + opt.best.x[1] ** 2 - 1.0))

### Convergence curves

Note the constraint-violation panel: the iterates start infeasible and the
filter/restoration machinery drives $\vartheta \to 0$ while the objective
decreases.

In [ ]:
viz.plot_convergence(opt, "Constrained convergence")
plt.show()

### Optimization path

The feasible region (unit disc) is shaded; the trajectory approaches the
constraint boundary where the constrained optimum lives (green cross =
SciPy reference solution).

In [ ]:
viz.plot_trajectory_2d(
    opt, cf,
    xlim=(-3, 3), ylim=(-3, 3),
    ineqs=[disc],
    solution=[0.7945, 0.6073],
    title="Constrained optimization path (feasible region shaded)",
)
plt.show()